# Skeleton of Thought (SoT) | Reasoning Patterns

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List
from concurrent.futures import ThreadPoolExecutor
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke
import json
import re

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
class SoTState(TypedDict):
    question: str
    skeleton: List[str]
    expanded_sections: List[str]
    final_answer: str

def generate_skeleton(state: SoTState) -> dict:
    """Generate a high-level outline/skeleton of the answer."""
    response = model.invoke(
        f"Generate a skeleton (outline) for answering this question. "
        f"List 4-6 key points as a JSON array of strings. Each point should be a section heading.\n\n"
        f"Question: {state['question']}\n\n"
        f"Return ONLY a JSON array of section headings."
    )
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", response.content).strip()
    try:
        skeleton = json.loads(cleaned)
    except json.JSONDecodeError:
        skeleton = [response.content]
    return {"skeleton": skeleton}

def expand_sections(state: SoTState) -> dict:
    """Expand each skeleton point in parallel."""
    def expand_point(args: tuple) -> str:
        idx, point = args
        response = model.invoke(
            f"You are writing section {idx + 1} of a detailed answer.\n\n"
            f"Overall question: {state['question']}\n"
            f"Full outline: {state['skeleton']}\n\n"
            f"Write section: '{point}'\n\n"
            f"Provide 2-3 detailed paragraphs for this specific section only."
        )
        return f"## {point}\n\n{response.content}"

    with ThreadPoolExecutor(max_workers=len(state["skeleton"])) as executor:
        sections = list(executor.map(expand_point, enumerate(state["skeleton"])))
    return {"expanded_sections": sections}

def assemble(state: SoTState) -> dict:
    """Assemble the expanded sections into a final answer."""
    assembled = "\n\n".join(state["expanded_sections"])
    return {"final_answer": assembled}

In [5]:
graph = StateGraph(SoTState)
graph.add_sequence([("skeleton", generate_skeleton), ("expand", expand_sections), ("assemble", assemble)])
graph.add_edge(START, "skeleton")
graph.add_edge("assemble", END)

sot = graph.compile()

In [6]:
# Plot the workflow
plot_mermaid(sot)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	skeleton(skeleton)
	expand(expand)
	assemble(assemble)
	__end__([<p>__end__</p>]):::last
	__start__ --> skeleton;
	expand --> assemble;
	skeleton --> expand;
	assemble --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [7]:
result = sot.invoke({
    "question": "Explain the complete lifecycle of an AI agent in production, from design to monitoring"
})
print(result["final_answer"])

## 1. Design and Planning

1. **Design and Planning**

The design and planning phase is the crucial foundation of the AI agent's lifecycle, fundamentally shaping its capabilities, functionality, and success. This stage begins with a thorough understanding of the problem space and specific business objectives the AI solution is intended to address. It's essential for stakeholders, including data scientists, project managers, domain experts, and end-users, to collaborate in defining the goals and constraints of the project. Clear articulation of the problem statement ensures that all parties have a unified vision of what the AI agent is expected to achieve. During this phase, stakeholders also work to identify and prioritize the key performance indicators (KPIs) that will determine the AI’s effectiveness. Understanding user needs and potential operational contexts will also guide the AI's design considerations.

Once the objectives are clearly defined, the focus shifts to data acquisitio

In [8]:
stream_invoke(sot, {
    "question": "Explain the complete lifecycle of an AI agent in production, from design to monitoring"
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'question': 'Explain the complete lifecycle of an AI agent in production, from design to monitoring',
 'skeleton': ['1. Design and Planning',
  '2. Development and Training',
  '3. Testing and Validation',
  '4. Deployment',
  '5. Monitoring and Maintenance',
  '6. Continuous Improvement'],
 'expanded_sections': ["## 1. Design and Planning\n\n1. **Design and Planning**\n\nThe design and planning phase is the crucial foundation of the AI agent's lifecycle, fundamentally shaping its capabilities, functionality, and success. This stage begins with a thorough understanding of the problem space and specific business objectives the AI solution is intended to address. It's essential for stakeholders, including data scientists, project managers, domain experts, and end-users, to collaborate in defining the goals and constraints of the project. Clear articulation of the problem statement ensures that all parties have a unified vision of what the AI agent is expected to achieve. During this pha